# 05_DeepResearch

## Planning Agent

In [1]:
# 导入一个你可以用的模型
from langchain_deepseek import ChatDeepSeek

api_key = ''
base_url = 'https://api.deepseek.com/v1'

model = ChatDeepSeek(model="deepseek-chat", api_key=api_key, base_url=base_url)

None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [2]:
from pydantic import BaseModel, Field

class WebSearchItem(BaseModel):
    """网页搜索项（单条搜索指令）"""
    query: str = Field(..., description="用于网页搜索的关键词，只写一个关键词")
    reason: str = Field(..., description="该搜索项对回答原问题的必要性说明")
    
class WebSearchPlan(BaseModel):
    """网页搜索计划（包含多条搜索指令）"""
    searches: list[WebSearchItem] = Field(..., description="为精准回答原问题需执行的网页搜索列表")

In [3]:
from langchain.agents import create_agent

# 定义 planning_system_prompt
planning_system_prompt = '''
你是专业研究助手，需基于用户查询完成以下核心任务，确保搜索精准高效：
1. 设计 10 个不重复的网页搜索关键词，关键词需贴合查询核心需求，兼顾精准度与覆盖性；
2. 为每个关键词单独说明搜索原因，说明该关键词如何助力解答用户查询，原因需具体可落地，不笼统。
'''

planning_agent = create_agent(model=model,
                     system_prompt=planning_system_prompt,
                     response_format = WebSearchPlan)

In [4]:
response = planning_agent.invoke(
    {"messages": [{"role": "user", "content": "AI在教育中的应用"}]}
)

search_plan = response["structured_response"]
queries = [item for item in search_plan.searches]

In [5]:
queries

[WebSearchItem(query='AI在教育中的应用', reason='全面了解AI在教育领域的整体应用场景和最新趋势，作为回答的基础框架'),
 WebSearchItem(query='人工智能 个性化学习 教育', reason='深入了解AI如何实现因材施教、个性化学习路径推荐等具体应用'),
 WebSearchItem(query='智能辅导系统 AI tutor', reason='搜索AI辅导系统（如智能家教、自动答疑）的具体案例和技术方案'),
 WebSearchItem(query='AI 自动批改 作业 考试 评分', reason='了解AI在作业批改、考试评分中的自动化应用及准确度'),
 WebSearchItem(query='自适应学习 AI 教育平台', reason='搜索自适应学习平台（如Knewton、猿辅导等）如何利用AI调整教学内容'),
 WebSearchItem(query='AI 虚拟教师 教育 应用', reason='了解虚拟教师、AI助教在课堂教学中的实际应用和效果'),
 WebSearchItem(query='教育数据挖掘 学习分析 AI', reason='研究AI如何通过数据分析预测学生学习表现、发现学习困难'),
 WebSearchItem(query='AI 特殊教育 辅助学习', reason='探索AI在特殊教育（如自闭症、读写障碍）中的辅助应用'),
 WebSearchItem(query='AI 教育 伦理 隐私 挑战', reason='了解AI在教育应用中面临的隐私、公平性、伦理等挑战和争议'),
 WebSearchItem(query='生成式AI ChatGPT 教育 应用案例', reason='搜索大语言模型（如ChatGPT）在教学设计、内容生成等方面的最新应用案例')]

## Execute Agent

In [6]:
from langchain_community.chat_models import ChatTongyi

qwen_search_model = ChatTongyi(
    model='qwen-max',
    api_key='',
    base_url='https://dashscope.aliyuncs.com/compatible-mode/v1',
    model_kwargs={
        # 开启联网搜索
        "enable_search": True,
    }
)

C:\Users\yanchen\AppData\Local\Temp\ipykernel_35200\66068312.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.chat_models import ChatTongyi


In [7]:
from langchain.agents import create_agent

execute_prompt = '''
你是一名专业的研究助理。给定一个搜索关键词后，需通过网络搜索该关键词，并生成简洁的搜索结果摘要。
摘要需包含 2-3 个段落，字数控制在 500 字以内，需涵盖核心要点。
表述需简洁凝练，无需使用完整句子或注重语法规范。
该摘要将供他人整合报告使用，因此务必提炼核心信息、剔除无关内容。除摘要本身外，不得添加任何额外评论。
'''

execute_agent = create_agent(
    system_prompt=execute_prompt,
    model=qwen_search_model,
)

execute_agent_response = execute_agent.invoke(
    {"messages": [{"role": "user", "content": "AI教育政策与发展趋势"}]}
)

In [8]:
execute_agent_response

{'messages': [HumanMessage(content='AI教育政策与发展趋势', additional_kwargs={}, response_metadata={}, id='ebfdbe3d-4468-4de7-a12a-ee057b00ab8e'),
  AIMessage(content='2026年中国AI教育政策框架已经明确，通过《"人工智能+教育"行动计划》等三份核心文件搭建起AI教育政策体系。这些政策旨在推动AI技术在教育领域的应用，促进智慧教育新形态的形成。行动计划特别强调了“做中学”的理念，并与创客教育硬件编程的教学实践相匹配。同时，《智能体规范应用与创新发展实施意见》对教育智能体的技术研发和应用范围进行了规范，确保数据安全与算法透明；《中国青少年阅读素养框架》则将AI阅读辅助工具纳入素养评估体系，重视深度阅读。\n\n根据最新政策要求，AI教育应用场景被分为禁止准入、有限使用及鼓励使用三大类进行管理。禁止准入类包括替代教师人格教育等12项行为红线，为教育科技产品研发划定了清晰边界。有限使用类如智能批改需经教师复核等8类场景，强调人机协同。而鼓励使用类涵盖智能学伴、特殊教育辅助等5大方向，为STEM教具开发提供了指引。此外，还确立了四大伦理原则：主体归人、适境致善、分类施治及动态调整，以保障AI教育应用健康发展。\n\n发展趋势方面，全球AI教育市场正经历从技术试点向系统部署的规模化转型，多模态AI智能体成为教育生态重构的关键引擎。技术创新推动市场增长，个性化学习成为主流趋势。AI教育不仅是短期流量噱头，而是教育行业数字化升级的必然趋势，具备极强的稳定性和成长性。随着国家持续鼓励“人工智能+教育”深度融合，智慧教育、数字化教学已成为教育行业核心发展方向。未来，AI赋能教育将在个性化学习、教学支持等方面有显著提升，但同时也面临着数据隐私、算法公平等挑战。构建可信赖的AI教育生态，关注教育公平，将是未来发展的重要方向。', additional_kwargs={}, response_metadata={'model_name': 'qwen-max', 'finish_reason': 'stop', 'request_id': 'a0b5183c-e996-971b-be72-8c13afe4a120', 'token_usag

In [9]:
search_content = execute_agent_response['messages'][-1].content
search_content

'2026年中国AI教育政策框架已经明确，通过《"人工智能+教育"行动计划》等三份核心文件搭建起AI教育政策体系。这些政策旨在推动AI技术在教育领域的应用，促进智慧教育新形态的形成。行动计划特别强调了“做中学”的理念，并与创客教育硬件编程的教学实践相匹配。同时，《智能体规范应用与创新发展实施意见》对教育智能体的技术研发和应用范围进行了规范，确保数据安全与算法透明；《中国青少年阅读素养框架》则将AI阅读辅助工具纳入素养评估体系，重视深度阅读。\n\n根据最新政策要求，AI教育应用场景被分为禁止准入、有限使用及鼓励使用三大类进行管理。禁止准入类包括替代教师人格教育等12项行为红线，为教育科技产品研发划定了清晰边界。有限使用类如智能批改需经教师复核等8类场景，强调人机协同。而鼓励使用类涵盖智能学伴、特殊教育辅助等5大方向，为STEM教具开发提供了指引。此外，还确立了四大伦理原则：主体归人、适境致善、分类施治及动态调整，以保障AI教育应用健康发展。\n\n发展趋势方面，全球AI教育市场正经历从技术试点向系统部署的规模化转型，多模态AI智能体成为教育生态重构的关键引擎。技术创新推动市场增长，个性化学习成为主流趋势。AI教育不仅是短期流量噱头，而是教育行业数字化升级的必然趋势，具备极强的稳定性和成长性。随着国家持续鼓励“人工智能+教育”深度融合，智慧教育、数字化教学已成为教育行业核心发展方向。未来，AI赋能教育将在个性化学习、教学支持等方面有显著提升，但同时也面临着数据隐私、算法公平等挑战。构建可信赖的AI教育生态，关注教育公平，将是未来发展的重要方向。'

## Synthesize Agent

In [10]:
synthesize_prompt = '''
你是一名资深研究员，负责为一项研究查询撰写结构连贯的报告。
你将收到原始查询需求以及研究助理完成的初步研究成果。
请首先制定报告大纲，明确报告的结构框架与逻辑脉络。
随后基于大纲撰写完整报告，并将其作为最终输出。
内容有有结构有段落，最底层的结构不要用1. 或者- 分点描述，直接用文字描述。
最终报告需采用 Markdown 格式，内容需详尽深入。
字数不少于 2000 字。
'''

class SynthesizeData(BaseModel):
    short_summary: str
    """研究结果的简短总结，200 字以内。"""

    markdown_report: str
    """最终报告"""

    follow_up_questions: list[str]
    """建议进一步研究的主题"""

In [11]:
# 使用当输入token大于10000时进行总结
from langchain.agents.middleware import SummarizationMiddleware

summary_prompt =  '''
你是一名资深研究员，负责为一项研究查询撰写结构连贯的报告。
目前有一份报告，你需要在不要过多的删减的情况下从现有报告中精确提炼重点。
'''

synthesize_agent = create_agent(
    system_prompt=synthesize_prompt,
    model=model,
    middleware=[SummarizationMiddleware(
        model=model,
        trigger=("tokens", 20000),
        summary_prompt=summary_prompt,
    )
               ],
    response_format=SynthesizeData   
)

synthesize_agent_response = synthesize_agent.invoke(
    {"messages": [{"role": "user", "content": search_content}]}
)

In [12]:
synthesize_agent_response

{'messages': [HumanMessage(content='2026年中国AI教育政策框架已经明确，通过《"人工智能+教育"行动计划》等三份核心文件搭建起AI教育政策体系。这些政策旨在推动AI技术在教育领域的应用，促进智慧教育新形态的形成。行动计划特别强调了“做中学”的理念，并与创客教育硬件编程的教学实践相匹配。同时，《智能体规范应用与创新发展实施意见》对教育智能体的技术研发和应用范围进行了规范，确保数据安全与算法透明；《中国青少年阅读素养框架》则将AI阅读辅助工具纳入素养评估体系，重视深度阅读。\n\n根据最新政策要求，AI教育应用场景被分为禁止准入、有限使用及鼓励使用三大类进行管理。禁止准入类包括替代教师人格教育等12项行为红线，为教育科技产品研发划定了清晰边界。有限使用类如智能批改需经教师复核等8类场景，强调人机协同。而鼓励使用类涵盖智能学伴、特殊教育辅助等5大方向，为STEM教具开发提供了指引。此外，还确立了四大伦理原则：主体归人、适境致善、分类施治及动态调整，以保障AI教育应用健康发展。\n\n发展趋势方面，全球AI教育市场正经历从技术试点向系统部署的规模化转型，多模态AI智能体成为教育生态重构的关键引擎。技术创新推动市场增长，个性化学习成为主流趋势。AI教育不仅是短期流量噱头，而是教育行业数字化升级的必然趋势，具备极强的稳定性和成长性。随着国家持续鼓励“人工智能+教育”深度融合，智慧教育、数字化教学已成为教育行业核心发展方向。未来，AI赋能教育将在个性化学习、教学支持等方面有显著提升，但同时也面临着数据隐私、算法公平等挑战。构建可信赖的AI教育生态，关注教育公平，将是未来发展的重要方向。', additional_kwargs={}, response_metadata={}, id='fff1fab0-475f-4dd6-8c73-e263ebb179e0'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 6291, 'prompt_tokens': 798, 'total_tokens': 7089, 'completion_tokens_detail

In [13]:
from rich.markdown import Markdown
synthesize_agent_response_summary = synthesize_agent_response["structured_response"].short_summary
synthesize_agent_response_report = synthesize_agent_response["structured_response"].markdown_report
synthesize_agent_response_follow_up_questions = synthesize_agent_response["structured_response"].follow_up_questions

# 渲染打印 markdown 报告
clean_markdown = synthesize_agent_response_report.strip("```").strip("markdown").strip()
display(Markdown(clean_markdown))

2026年中国AI教育政策框架与产业发展深度研究报告                                   

一、报告大纲                                                                                                       

第一章 研究背景与政策全景                                                                                          

 • 1.1 研究背景                                                                                                    
 • 1.2 三份核心政策文件概览                                                                                        
 • 1.3 政策体系的总体定位与战略意义                                                                                

第二章 核心政策文件深度解读                                                                                        

 • 2.1 《"人工智能+教育"行动计划》                                                                                 
 • 2.2 《智能体规范应用与创新发展实施意见》                                                                        
 • 2.3 《中国青少年阅读素养框架》                                                                                  

第三章 三分类准入管理体系                                                                                          

 • 3.1 分类管理机制的总体设计                                                                                      
 • 3.2 禁止准入类（12项红线）                                                                                      
 • 3.3 有限使用类（8类场景与人机协同要求）                                                                         
 • 3.4 鼓励使用类（5大方向）                                                                                       

第四章 四大伦理原则体系                                                                                            

 • 4.1 主体归人原则                                                                                                
 • 4.2 适境致善原则                                                                                                
 • 4.3 分类施治原则                                                                                                
 • 4.4 动态调整原则                                                                                                

第五章 全球AI教育市场发展趋势                                                                                      

 • 5.1 市场规模与增长态势                                                                                          
 • 5.2 从技术试点到系统部署的转型                                                                                  
 • 5.3 多模态AI智能体的核心驱动作用                                                                                
 • 5.4 个性化学习的主流化趋势                                                                                      

第六章 中国AI教育产业机遇与挑战                                                                                    

 • 6.1 政策红利下的产业机遇                                                                                        
 • 6.2 数据隐私与算法公平挑战                                                                                      
 • 6.3 教育公平与数字鸿沟问题                                                                                      

第七章 未来展望与战略建议                                                                                          

 • 7.1 可信赖AI教育生态构建路径                                                                                    
 • 7.2 面向教育公平的政策优化建议                                                                                  
 • 7.3 产学研协同创新方向                                                                                          

-------------------------------------------------------------------------------------------------------------------

第二章 研究背景与政策全景                                                                                          

1.1 研究背景                                                                                                       

2026年，中国人工智能教育政策体系迎来了里程碑式的构建完成。在数字经济与教育现代化深度融合的大背景下，国家通过三份核 
心政策文件——即《"人工智能+教育"行动计划》、《智能体规范应用与创新发展实施意见》以及《中国青少年阅读素养框架》——搭建
起了完整的AI教育政策框架。这一政策体系的出台，标志着中国AI教育从零散探索阶段正式迈入系统化、制度化的发展新周期。   

从宏观层面看，这一政策体系的构建与国家对教育强国、科技强国建设的战略布局

## deepResearchManager

In [15]:
import os


class deepResearchManager:
    def __init__(self):
        print("初始化已完成，欢迎使用。\n使用前请确认相关模型能够被顺利调用。")

    def run(self, query: str) -> None:

        print("Starting research...")
        # Step 1: 根据用户 qury 生成检索计划
        search_plan = self.plan_searches(query)
        
        # Step 2: 根据生成检索计划进行网页搜索
        search_results = self.perform_searches(search_plan)
        
        # Step 3: 根据检索内容生成 markdown 格式的报告
        markdown_content, follow_up_questions = self.write_report(query, search_results)

        print("\n\n=====REPORT=====\n\n")
        display(Markdown(markdown_content))
        print("\n\n=====FOLLOW UP QUESTIONS=====\n\n")
        
        follow_up_questions_text = "\n".join(follow_up_questions)
        display(Markdown(follow_up_questions_text))
        
        # 保存为 Markdown 文件
        self.save_report_as_md(query, markdown_content)
    
    def plan_searches(self, query: str) -> WebSearchPlan:
        print("Planning searches...")
        response = planning_agent.invoke(
            {"messages": [{"role": "user", "content": query}]}
        )
        return response

    def perform_searches(self, search_plan: WebSearchPlan) -> list[str]:
        print("Starting searching...")
        num_completed = 0
        search_plan = search_plan["structured_response"]
        tasks = [item.query for item in search_plan.searches]

        results = []
        for task in tasks:
            search_agent_res = execute_agent.invoke({"messages": [{"role": "user", "content": task}]})
            search_content = search_agent_res['messages'][-1].content
            
            if search_content is not None:
                results.append(search_content)
            num_completed += 1
            print(f"Searching... {num_completed}/{len(tasks)} completed")
            
        return results

    def write_report(self, query: str, search_results: list[str]) -> tuple[str, list[str]]:
        print("Thinking about report...")
        write_report_query = f": 用户的需求是：{query}\n汇总的搜索结果如下: {search_results}"
        
        synthesize_agent_response = synthesize_agent.invoke(
            {"messages": [{"role": "user", "content": write_report_query}]})

        synthesize_agent_response_summary = synthesize_agent_response["structured_response"].short_summary
        synthesize_agent_response_report = synthesize_agent_response["structured_response"].markdown_report
        # 处理Markdown内容：去掉代码块的 ``` 标记
        markdown_content = synthesize_agent_response_report.strip("```").strip("markdown").strip()
        synthesize_agent_response_follow_up_questions = synthesize_agent_response["structured_response"].follow_up_questions
        return markdown_content, synthesize_agent_response_follow_up_questions

    def save_report_as_md(self, query: str, markdown_content: str) -> None:
        """
        保存生成的报告为 Markdown 文件
        """
        # 使用用户的查询作为文件名
        sanitized_query = query.replace(" ", "_").replace("：", "").replace("?", "")
        
        file_name = f"关于{sanitized_query}调研报告.md"
        file_path = os.path.join(os.getcwd(), file_name)
        with open(file_path, "w", encoding="utf-8") as file:
            file.write(markdown_content)
            
        print(f"Report saved as: {file_name}")

In [16]:
manager = deepResearchManager()

初始化已完成，欢迎使用。
使用前请确认相关模型能够被顺利调用。


In [17]:
test_query = "AI在教育中的应用"

In [18]:
manager.run(test_query)

Starting research...
Planning searches...
Starting searching...
Searching... 1/10 completed
Searching... 2/10 completed
Searching... 3/10 completed
Searching... 4/10 completed
Searching... 5/10 completed
Searching... 6/10 completed
Searching... 7/10 completed
Searching... 8/10 completed
Searching... 9/10 completed
Searching... 10/10 completed
Thinking about report...


=====REPORT=====




AI在教育中的应用：全面研究报告                                           

一、引言                                                                                                           

2024年被广泛视为"AI+教育"的变革之年，人工智能与教育的融合达到了前所未有的高度。从政策层面到技术实践，从课堂教学到教
育管理，AI正在以多种方式重塑教育的形态与内涵。本报告旨在系统梳理AI在教育领域的应用现状、核心场景、技术支撑、面临的 
挑战以及未来发展趋势，为教育工作者、政策制定者及技术开发者提供全面而深入的参考。                                   

二、报告大纲                                                                                                       

 1 引言：问题背景与研究意义                                                                                        
 2 政策环境与战略布局：国内外AI教育政策演进                                                                        
 3 核心技术支撑：支撑AI教育应用的关键技术                                                                          
 4 AI教育的核心应用场景                                                                                            
    • 个性化学习与自适应教育                                                                                       
    • 智能辅导系统                                                                                                 
    • 沉浸式学习与教育游戏                                                                                         
    • 自动批改与智能评分                                                                                           
    • 教育管理与学情分析                                                                                           
    • 语言学习与口语评测                                                                                           
 5 伦理挑战与数据隐私                                                                                              
 6 教师角色的转变与人机协作                                                                                        
 7 未来展望与发展趋势                                                                                              
 8 结论                                                                                                            

三、政策环境与战略布局                                                                                             

3.1 国家层面的政策推动                                                                                             

近年来，中国在"人工智能+教育"领域持续发力，政策体系日趋完善。2024年，教育部正式启动"人工智能赋能教育行动"，旨在探索
人工智能教育的有效路径。该行动不仅关注技术的应用，还强调通过政策引导确保人工智能在教育中的健康发展。同年，政府工作 
报告明确提出深化人工智能研发应用，教育部积极推进打造中国版的人工智能教育大模型。                                   

更为重要的是，2026年教育部等五部门联合发布的《"人工智能+教育"行动计划》为行业发展指明了方向。该计划明确了四大重点任
务：一是加强人工智能人才培养与素养提升；二是促进人工智能与教育深度融合；三是建设强大的"人工智能+教育"基础环境；四是
打造开放的"人工智能+教育"生态系统。这些措施旨在建立一个涵盖从基础教育到高等教育、职业教育乃至终身教育阶段的人工智能
全学段教育体系。                                                                                                   

3.2 地方实践与标准制定                                                                                             

在地方层面，北京市率先发布了《北京市教育领域人工智能应用指南（2024年）》，明确了人工智能在学校教育中的六大应用领域—
—助教、助学、助评、助育、助研、助管，并详细列出29个典型应用场景，为学校稳妥有序地开展相关实践提供了操作指引。      

与此同时，《关于深入实施"人工智能+"行动的意见》等政策文件强调要将人工智能融入教育教学全要素全过程，促进教育资源均衡
分配，并利用AI技术缩小城乡、区域间的教育差距，推动教育公平与质量提升。国家智慧教育公共服务平台上已汇聚超过1000门人 
工智能精品课程，满足不同学段人群的学习需求，体现了全学段和社会通识教育全覆盖的目标。                               

四、核心技术支撑                                                                                                   

4.1 深度学习与自然语言处理                                                                                         

深度学习技术的突破性进展为AI教育的智能化提供了核心动力。通过多层神经网络架构，深度学习模型能够从海量教育数据中自动 
提取特征，实现对学生学习行为的精准建模。自然语言处理技术的进步则使得AI系统能够理解、生成和评估自然语言文本，为智能 
批改、对话式辅导、语言学习等应用场景提供了技术基础。                                                               

4.2 语音识别与合成                                                                                                 

语音识别技术是AI口语评测系统的核心技术支撑。通过声学模型、语言模型和发音词典的协同工作，AI系统能够准确识别学习者的 
语音内容，并对其发音准确性、流利度、语调自然度等进行多维度的自动化评估。语音合成技术则让AI系统能够以自然的人声与学 
习者进行互动，提升人机交互的体验感。                                                                               

4.3 虚拟现实与增强现实                                                                                             

虚拟现实（VR）、增强现实（AR）以及混合现实（MR）技术的引入，为学



=====FOLLOW UP QUESTIONS=====




AI教育应用在不同学段（小学、中学、大学）的差异化策略如何制定？                                                     
如何建立有效的AI教育效果评估体系，衡量其对学习成果的实际影响？                                                     
特殊教育领域中，AI技术如何更好地服务于有特殊需求的学生群体？                                                       
AI教育的成本效益分析——中小学校引入AI教育系统的经济可行性如何？                                                     
在AI辅助教育的背景下，教师培训体系应如何进行适应性改革？

Report saved as: 关于AI在教育中的应用调研报告.md
